# L99 - Standalone Graph Runtime Pack 0.1.2 Readiness Fix - Lemma Proof Bundle v1

**Lemma id:** `L99_m365_standalone_graph_runtime_pack_0_1_2_readiness_fix_v1`
**Plan reference:** `plan:m365-standalone-graph-runtime-pack-0-1-2-readiness-fix:C2`
**Module notebook:** `notebooks/m365/INV-M365-DO-standalone-graph-runtime-pack-0-1-2-readiness-fix-v1.ipynb`
**Mode:** deterministic-static (no random sampling, no live network).

## Lemmas under proof
- `L_VERSION_CORRECT` - active manifest, runtime constant, runtime endpoint, bundle filename, and install path all read `0.1.2`; historical `1.1.x` references stay labeled-historical only.
- `L_ARTIFACT_SELFDESC` - `probe_artifact()` returns `True` from the chosen installed root after extracting the outer `.ucp.tar.gz` and inner `payload.tar.gz`, with no source/sibling/`M365_REPO_ROOT` reads.
- `L_DEPENDENCY_CLOSED` - declared dependencies are recorded in the pack; missing dependencies surface as `dependency_missing` with module names; no implicit `ModuleNotFoundError`; no accidental `repo .venv` reuse.
- `L_SOCKET_REAL` - release acceptance launches the runtime as a subprocess on a loopback dynamic port and calls `ucp_m365_pack.client.execute_m365_action()` over real `httpx` without monkey-patching `_http_runtime_invoke`; guard fails the run if patching is detected.
- `L_PROVENANCE_REPRO` - two consecutive deterministic builds at the recorded source commit produce byte-identical bundle SHAs; provenance records source commit/branch/clean state, dependency lock SHA, payload SHA, bundle SHA, manifest SHA, and conformance SHA; final `GO` requires `clean == True`.

Bundled predicate:

`PackReady_0_1_2 = L_VERSION_CORRECT AND L_ARTIFACT_SELFDESC AND L_DEPENDENCY_CLOSED AND L_SOCKET_REAL AND L_PROVENANCE_REPRO`


In [ ]:
# Symbolic re-statement of the lemma predicates from INV-M365-DO. We keep this
# notebook minimal because the module notebook already enumerates the green
# base case and 17 negative cases; the role of this lemma proof is to bind
# each named lemma to its exact failure boundary.
import json
from pathlib import Path

FAILURE_BOUNDARIES = {
    'L_VERSION_CORRECT': [
        'manifest declares any version other than 0.1.2',
        'runtime RUNTIME_VERSION constant differs from 0.1.2',
        'runtime /v1/runtime/version returns any value other than 0.1.2',
        'bundle filename or install dir uses 1.1.x path',
        'an active release authority document still claims 1.1.x as ready',
    ],
    'L_ARTIFACT_SELFDESC': [
        'launching from chosen installed root finds probe_artifact()=False',
        'runtime resolves any required artifact through source repo path',
        'runtime resolves any required artifact through sibling repo path',
        'runtime reads M365_REPO_ROOT or SMARTHAUS_M365_REPO_ROOT at runtime',
        'required metadata file missing from the installed payload root',
    ],
    'L_DEPENDENCY_CLOSED': [
        'dependency manifest/lock missing from the installed payload',
        'startup probe absent or runs after listener bind',
        'missing module surfaces as raw ModuleNotFoundError instead of dependency_missing',
        'pack quietly inherits repo .venv site-packages',
        'optional JWT/cert path blocks delegated device_code import without declaration',
    ],
    'L_SOCKET_REAL': [
        'release acceptance uses TestClient or in-process transport instead of subprocess',
        '_http_runtime_invoke is monkey-patched in release acceptance',
        'release acceptance fails to set M365_RUNTIME_URL',
        'execute_m365_action() bypasses real httpx call to local socket',
        'no legacy alias mapping is exercised over real HTTP',
    ],
    'L_PROVENANCE_REPRO': [
        'two consecutive deterministic builds produce different bundle SHAs',
        'provenance.json missing one of source_commit, source_branch, source_clean, dependency_lock_sha, payload_sha, bundle_sha, manifest_sha, conformance_sha',
        'provenance claims clean while git tree is dirty',
        'recorded commit cannot reproduce the artifact bytes',
    ],
    'PackReady_0_1_2': [
        'any single lemma above evaluates to False',
    ],
}

for k, v in FAILURE_BOUNDARIES.items():
    assert v, f'lemma {k} must list at least one failure boundary'
    assert all(isinstance(s, str) and s for s in v), f'lemma {k} failure boundaries must be non-empty strings'

len(FAILURE_BOUNDARIES)


In [ ]:
# Tie each lemma to the test/script that mechanically enforces it.
TEST_BINDINGS = {
    'L_VERSION_CORRECT': [
        'tests/test_m365_runtime_p7_packaging.py::test_runtime_version_constant',
        'scripts/ci/verify_standalone_graph_runtime_pack.py - manifest+runtime version check',
        'tests/test_m365_runtime_0_1_2_readiness_fix.py::test_runtime_version',
    ],
    'L_ARTIFACT_SELFDESC': [
        'tests/test_m365_runtime_0_1_2_readiness_fix.py::test_installed_payload_art_true',
        'scripts/ci/acceptance_standalone_graph_runtime_pack.py - extract+launch+probe_artifact',
    ],
    'L_DEPENDENCY_CLOSED': [
        'tests/test_m365_runtime_0_1_2_readiness_fix.py::test_dependency_manifest_present',
        'tests/test_m365_runtime_0_1_2_readiness_fix.py::test_dependency_missing_outcome_class',
    ],
    'L_SOCKET_REAL': [
        'scripts/ci/acceptance_standalone_graph_runtime_pack.py - subprocess + real httpx + execute_m365_action()',
        'tests/test_m365_runtime_0_1_2_readiness_fix.py::test_acceptance_rejects_http_runtime_invoke_patch',
    ],
    'L_PROVENANCE_REPRO': [
        'scripts/ci/build_standalone_graph_runtime_pack.py - deterministic gzip mtime=0',
        'scripts/ci/acceptance_standalone_graph_runtime_pack.py - two-build SHA equality',
        'tests/test_m365_runtime_0_1_2_readiness_fix.py::test_provenance_required_keys',
    ],
}

assert set(TEST_BINDINGS.keys()) == {k for k in FAILURE_BOUNDARIES.keys() if k != 'PackReady_0_1_2'}
len(TEST_BINDINGS)


In [ ]:
# Symbolic conjunction proof for PackReady_0_1_2. Each lemma is a boolean.
# We enumerate the truth table over 5 booleans (32 rows) and prove the
# conjunction is True only when all five are True.
from itertools import product

rows = []
for vals in product([False, True], repeat=5):
    v, a, d, s, p = vals
    pack_ready = v and a and d and s and p
    rows.append((vals, pack_ready))

true_rows = [r for r in rows if r[1] is True]
false_rows = [r for r in rows if r[1] is False]
assert len(rows) == 32
assert len(true_rows) == 1
assert true_rows[0][0] == (True, True, True, True, True)
assert len(false_rows) == 31
(len(true_rows), len(false_rows))
